# d_health: end-to-end model run

This notebook generates `config.toml` from Python variables you can edit below, then runs the full health-impact pipeline once the four model inputs are in place:

1. **Flood depth raster** — the scenario input (netCDF or GeoTIFF).
2. **Population netCDF** — a labelled `group` dimension (`children`, `adults`, `total`).
3. **Urban/rural raster** — `1 = urban`, `2 = rural`, `0 = nodata`.
4. **Country indicators TOML** — GDP per capita + sanitation breakdown for the country.

The first three are produced by the package's `preprocessing` routines (see `1_worldpop_pre_processing.ipynb` and `2_ghs_smod_pre_processing.ipynb`); the fourth is built in `3_world_bank_pre_processing.ipynb` via `build_from_wdi`. The model is agnostic to how those files were produced — you can hand-write a TOML matching `d_health/config/emissions.py::CountryIndicators` instead.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import tomli_w

from d_health import run_model_from_toml
from d_health.config import EventConfig, ExposureConfig, OutputConfig, RunConfig, SettingsConfig

## 1. Configure the run

Build the config sections as pydantic models — typos and missing fields fail fast:

- `ExposureConfig` — the region's static inputs: `population`, `urban_rural`, `country_indicators`.
- `EventConfig` — the forcing: the `flood_depth_map`.
- `OutputConfig` — `out_dir` + `plots`.

The next cell dumps these (plus any `[settings]` overrides) into `config.toml` inside `out_dir`. Model parameters (pathogen dose-response, population groups, emissions constants) each have a built-in default, so `[settings]` is optional — declare fields here only to override.

In [ ]:
# The region's static inputs, produced by the pre-processing notebooks (0–3).
exposure = ExposureConfig(
    population='data/sur_population_2020_combined.nc',
    urban_rural='data/sur_urban_rural.nc',
    country_indicators='data/sur_indicators.toml',
)

# The forcing: the flood scenario raster.
event = EventConfig(flood_depth_map='data/flood_depth_paramaribo.tif')

# Where model outputs land (rasters + plots).
output_cfg = OutputConfig(
    out_dir='outputs/run',
    plots=True,
)

# Pathogen — defaults to the bundled E.coli entry; override here to change it.
pathogen = 'E.coli'

## 2. Write `config.toml`

Serialise the pydantic models above into a dict and dump it with `tomli_w` into `<out_dir>/config.toml`. The file is the only thing `run_model_from_toml` reads, so anything not declared under `[settings]` falls through to the bundled object defaults. Extend `run_dict['settings']` with `population_groups`, `emissions`, etc. when you need to override more.

In [ ]:
# Create a SettingsConfig to override just the pathogen, keeping all other defaults
settings = SettingsConfig(pathogen={'selected': pathogen})

# Build the complete RunConfig from the config objects
run_config = RunConfig(
    exposure=exposure,
    event=event,
    settings=settings,
    output=output_cfg,
)

# Serialize to dict with all parameters (including defaults) in JSON-compatible format
run_dict = run_config.model_dump(mode='json')

# Create output directory
out_dir = Path(output_cfg.out_dir)
out_dir.mkdir(parents=True, exist_ok=True)
config_toml_path = out_dir / 'config.toml'

# Resolve all paths to absolute first to compute the relative path correctly
abs_population = exposure.population.resolve()
abs_urban_rural = exposure.urban_rural.resolve()
abs_country_indicators = exposure.country_indicators.resolve()
abs_flood_depth_map = event.flood_depth_map.resolve()
abs_config_dir = config_toml_path.parent.resolve()

# Compute relative paths from the config.toml location (allowing walk_up with ..)
run_dict['exposure']['population'] = str(abs_population.relative_to(abs_config_dir, walk_up=True))
run_dict['exposure']['urban_rural'] = str(abs_urban_rural.relative_to(abs_config_dir, walk_up=True))
run_dict['exposure']['country_indicators'] = str(abs_country_indicators.relative_to(abs_config_dir, walk_up=True))
run_dict['event']['flood_depth_map'] = str(abs_flood_depth_map.relative_to(abs_config_dir, walk_up=True))
run_dict['output']['out_dir'] = '.'  # Current directory (the config's parent directory)

with config_toml_path.open('wb') as f:
    tomli_w.dump(run_dict, f)

print(config_toml_path.read_text())

## 3. Run the model

`run_model_from_toml` loads the config (applying the default model settings and reading the country indicators TOML), runs the pipeline end-to-end, writes per-cell rasters and per-group plots to `output.out_dir`, and returns a `ModelOutputs` with everything in memory too.

In [ ]:
outputs = run_model_from_toml(config_toml_path)
outputs.totals

## 4. Inspect outputs inline

`ModelOutputs` exposes every array keyed by group name — easy to plot inline or hand off to your own analysis.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].imshow(outputs.emissions, cmap='viridis');     axes[0].set_title('E. coli emissions (CFU)')
axes[1].imshow(outputs.pathogen_conc, cmap='magma');   axes[1].set_title('Concentration (CFU/100 mL)')
axes[2].imshow(outputs.infected['adults'], cmap='Greens'); axes[2].set_title('Infected adults')
for ax in axes: ax.axis('off')
plt.tight_layout(); plt.show()

## 5. Coverage breakdown

`ModelOutputs.coverage` carries the flooded-vs-dry summary plus per-flood-class population counts. Class codes come from the union of every group's depth-threshold `min_depth` values — for the default adults/children groups, that's 0.1 / 0.5 / 1.5 m, giving a 4-class scheme (1 = minimum flooded, 2 = all wading, 3 = children swim / adults wade, 4 = both swimming).

In [ ]:
from d_health.config.loaders import load_run_config
from d_health.postprocessing import flood_class_labels

cfg = load_run_config(config_toml_path)
labels = flood_class_labels(cfg.settings.population_groups)

cov = outputs.coverage
total = cov.total['total']
flooded = cov.flooded['total']
print(f'Total inhabitants    : {round(total):>8d}')
print(f'In flooded area      : {round(flooded):>8d} ({flooded/total*100:.0f}%)')
print(f'In dry area          : {round(cov.dry["total"]):>8d} ({cov.dry["total"]/total*100:.0f}%)')
print()
for name in cov.flooded:
    if name == 'total':
        continue
    share = cov.flooded[name] / cov.total[name] * 100 if cov.total[name] else 0
    print(f'  {name:<20s} in flooded area: {round(cov.flooded[name]):>8d} ({share:.0f}%)')
print()
for cls in sorted(cov.per_class):
    entry = cov.per_class[cls]
    share = entry['total'] / total * 100 if total else 0
    print(f'Class {cls} ({labels[cls-1]:>35s}) : {round(entry["total"]):>8d} ({share:.0f}%)')

## 6. Flood-class map

A green/yellow/orange/red ramp by increasing flood severity. Class boundaries are derived from the population groups, not hard-coded — adding an `elderly` group with different depth thresholds refines the map automatically.

In [ ]:
from matplotlib.colors import BoundaryNorm, ListedColormap
import numpy as np

labels = flood_class_labels(cfg.settings.population_groups)
n = len(labels)
cmap = ListedColormap(['green', 'yellow', 'orange', 'red'][:n])
boundaries = [0.5] + [i + 1.5 for i in range(n)]
norm = BoundaryNorm(boundaries, cmap.N)

masked = np.ma.masked_where(outputs.flood_classes == 0, outputs.flood_classes)

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(masked, cmap=cmap, norm=norm)
cbar = fig.colorbar(im, ax=ax, ticks=range(1, n + 1), boundaries=boundaries)
cbar.ax.set_yticklabels(labels)
ax.set_title('Flood depth classes')
ax.axis('off')
plt.tight_layout(); plt.show()

## 7. Risk-class histogram

Population per risk bin, per group. `risk_class_edges` are the bin boundaries (default 0.0–0.2–0.4–0.6–0.8–1.0); change them with the `edges` kwarg of `bin_population_by_risk` if you want a different breakdown.

In [ ]:
import numpy as np

edges = outputs.risk_class_edges
n_bins = len(edges) - 1
n_groups = len(outputs.risk_class_counts)
index = np.arange(n_bins)
bar_width = 0.8 / n_groups

fig, ax = plt.subplots(figsize=(9, 5))
for i, (name, vals) in enumerate(outputs.risk_class_counts.items()):
    ax.bar(index + i * bar_width, vals, bar_width, label=name)

ax.set_xlabel('Risk class')
ax.set_ylabel('Population count')
ax.set_title('Population count in risk classes')
ax.set_xticks(index + bar_width * (n_groups - 1) / 2)
ax.set_xticklabels([f'{edges[i]:.1f}-{edges[i+1]:.1f}' for i in range(n_bins)])
ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
# Written artifacts on disk:
for name, path in outputs.paths.items():
    print(f'{name:>22}  {path}')